# 00. 선행연구 기반 K-ETS VWAP 타겟 정책 검토

이 문서는 K-ETS 가격 예측에서 사용할 VWAP 타겟 계열과 빈티지 처리 원칙을 정리한다. \
현재 후속 노트북 흐름은 날짜별 전체 거래 `market_vwap`을 주 타겟으로 사용한다. 같은 날짜에 거래된 모든 KAU vintage의 거래대금 합계를 거래량 합계로 나누어 시장 가격을 만든다. \
빈티지 전환 규칙은 단일-vintage continuous series를 만들 때 필요한 보조 정책으로 남긴다. `daily_max_volume`은 빈번한 앞뒤 전환을 확인하는 진단 항목으로만 둔다.

데이터 생성과 품질 검증은 `01_market_vwap_target.ipynb`, `02_target_eda.ipynb`, feature 점검은 `03_feature_readiness.ipynb`, `04_feature_target_eda.ipynb`에서 수행한다.


## 1. 용어 정의

| 용어 | 본 프로젝트에서의 사용 의미 |
|---|---|
| `market_vwap` | 같은 거래일에 실제 거래된 KAU vintage들의 `acc_trdval` 합계를 `acc_trdvol` 합계로 나눈 날짜별 전체 거래 VWAP이다. 후속 노트북의 주 타겟 가격 계열이다. |
| `vintage` | 배출권의 이행연도 또는 발행연도 성격을 갖는 구분이다. KRX 데이터에서는 `KAU15`, `KAU16` 같은 `isu_code`로 표현된다. |
| `blended day` | 한 날짜의 `market_vwap`가 2개 이상 vintage 거래를 합쳐 계산된 날이다. `is_blended`, `n_vintages`, `vintages`로 표시한다. |
| `same-vintage panel` | 같은 `isu_code` 내부에서만 미래 수익률을 계산하는 패널형 타겟 구성 방식이다. 현재 흐름에서는 강건성·진단용 비교 축으로 둔다. |
| `continuous series` | 여러 만기 또는 여러 빈티지의 가격을 일정한 전환 규칙으로 이어 붙여 만든 단일 시계열이다. |
| `roll rule` | 만기 또는 대표 계약을 다음 계약으로 넘기는 규칙이다. K-ETS 빈티지 구조에서는 **대표 빈티지 전환 규칙**이라고 표기한다. |
| `daily_max_volume` | 매일 거래량이 가장 큰 vintage를 대표로 선택하는 진단 지표이다. 앞뒤 전환이 잦으면 타겟 후보에서 제외한다. |

KRX KAU vintage는 선물 만기 계약과 상품 구조가 다르다. 따라서 후속 노트북은 먼저 `market_vwap`로 시장 단위 가격을 만들고, 빈티지 전환 규칙은 별도 민감도 분석 주제로 남긴다.


## 2. 문헌 사례 1: EU-ETS 연구에서의 현물-선물 가격 관계

EU-ETS는 K-ETS보다 거래 유동성이 높고 선물시장이 발달해 있다. EU-ETS 선행연구에서는 EUA 현물 가격과 EUA 선물 가격의 관계가 주요 연구 대상이 된다. Arouri, Jawadi, Nguyen(2012)은 EU-ETS Phase II의 탄소 현물-선물 가격 관계에서 비선형성을 분석하였다. 이 연구는 탄소 가격 시계열을 다룰 때 현물 가격, 선물 계약, 계약 간 연결 구조를 함께 검토해야 함을 보여준다.

K-ETS의 `KAU15`, `KAU16`, `KAU17` 등은 EU-ETS 선물 계약과 동일한 상품 구조를 갖지 않는다. 그러나 예측 타겟을 만들 때 여러 가격 계열을 어떤 규칙으로 통합할지 정해야 한다는 점에서 유사한 문제가 있다. 후속 노트북은 거래일 단위 전체 시장 가격인 `market_vwap`을 먼저 만들고, 빈티지 단위 선택 규칙은 진단 항목으로 확인한다.

| 문헌 | 확인 정보 | 본 프로젝트에 주는 시사점 |
|---|---|---|
| Arouri, Jawadi, Nguyen (2012), *Economic Modelling* | `Nonlinearities in carbon spot-futures price relationships during Phase II of the EU ETS`, DOI: `10.1016/j.econmod.2011.11.003` | 탄소 가격은 여러 가격 계열의 연결 구조와 함께 해석된다. K-ETS에서는 날짜별 전체 거래 VWAP와 vintage 단위 보조 진단을 구분한다. |


## 3. 문헌 사례 2: EUA 선물 가격을 직접 예측 대상으로 사용하는 연구

EUA 선물 가격 변동성 예측 연구는 특정 선물 가격 계열을 분석 대상으로 삼는다. Guo et al.(2022)은 EUA futures 변동성을 예측 대상으로 설정했고, Wu, Yin, Mei(2022)는 EUA futures 변동성에 기후정책 불확실성을 결합하였다. 두 사례 모두 예측 대상이 되는 가격 계열을 먼저 정한 뒤, 그 계열의 변동성 또는 수익률을 모델링한다.

K-ETS에서는 `raw.krx_ets_daily`의 `vwap`, `acc_trdvol`, `acc_trdval`, `isu_code`를 이용해 가격 계열을 구성한다. 현재 후속 노트북은 `sum(acc_trdval) / sum(acc_trdvol)` 기준의 날짜별 `market_vwap`를 주 타겟 가격으로 사용한다. `vwap * acc_trdvol` 기반 계산값은 raw VWAP 반올림 차이를 확인하는 check column으로 둔다.

| 문헌 | 확인 정보 | 본 프로젝트에 주는 시사점 |
|---|---|---|
| Guo et al. (2022), *Energy Economics* | `Forecasting volatility of EUA futures: New evidence`, DOI: `10.1016/j.eneco.2022.106021` | 예측 대상 가격 계열을 명시한 뒤 수익률·변동성을 계산한다. K-ETS에서는 `market_vwap` 정의가 타겟 정의의 출발점이다. |
| Wu, Yin, Mei (2022), *Sustainability* | `Forecasting the Volatility of European Union Allowance Futures with Climate Policy Uncertainty Using the EGARCH-MIDAS Model`, DOI: `10.3390/su14074306` | 일별 시장 가격과 저빈도 정책·불확실성 변수를 결합할 수 있다. K-ETS의 30일/60일 예측 기간도 외생 변수 주기와 함께 관리한다. |


## 4. 문헌 사례 3: continuous futures price series 방법론

선물시장에서는 여러 만기 계약을 이어 붙여 장기 가격 시계열을 만들 때 continuous futures price series를 사용한다. Went(2010)는 continuous futures price series를 만들 때 계약 전환 시점과 가격 조정 방식이 결과 시계열에 영향을 준다는 점을 정리한다. 핵심은 전환 규칙을 분석 전에 명시하고, 전환 뒤 되돌림과 가격 점프를 점검하는 것이다.

K-ETS의 KAU vintage는 선물 만기 계약과 법적 구조가 다르다. 여러 `isu_code`를 하나의 대표 vintage 시계열로 이어 붙일 때는 continuous futures 구성과 유사한 문제가 발생한다. 따라서 대표 vintage 전환 규칙을 추가 타겟으로 검토할 경우 다음 조건을 적용한다.

1. 전환 기준을 사전에 정의한다.
2. 전환 후 이전 vintage로 되돌아가는 현상을 제한한다.
3. 전환 시점 전후의 가격 점프를 별도로 진단한다.
4. 거래량, 거래대금, 거래 공백 등 시장 유동성 정보를 함께 확인한다.

| 문헌 | 확인 정보 | 본 프로젝트에 주는 시사점 |
|---|---|---|
| Went (2010), SSRN | `Creating Continuous Futures Price Series`, DOI: `10.2139/ssrn.1547310` | 여러 계약을 연결할 때 roll rule이 시계열의 성격을 결정한다. K-ETS 대표 vintage series는 주 타겟과 분리해 민감도 분석으로 관리한다. |


## 5. 프로젝트 레퍼런스 문서에서 확인한 K-ETS 특성

프로젝트 내부 레퍼런스 문서에서도 K-ETS는 EU-ETS와 다른 구조를 갖는다고 정리되어 있다. `가격_모델.md`는 EU-ETS의 시장 성숙도와 유동성이 높고 선물 및 파생상품 중심인 반면, K-ETS는 장내 현물 거래 비중이 높고 유동성 부족이 존재한다고 기술한다. 또한 K-ETS의 빈티지는 이월 및 차입 규정과 연결되며, 특정 빈티지의 이행기간 종료나 이월 제한 조치로 인해 빈티지별 특이 현상이 발생할 수 있다고 정리되어 있다.

이 특성은 타겟 가격 계열을 설계할 때 중요하다. 날짜별 시장 가격은 전체 거래대금과 거래량을 합산한 `market_vwap`로 구성한다. 빈티지별 제도 효과와 겹침 구간은 `n_vintages`, `vintages`, `is_blended`, `vwap_range` 같은 보조 컬럼으로 남긴다.

| 출처 | 확인 내용 | 타겟 설계 반영 |
|---|---|---|
| `docs/reference/가격_모델.md` | EU-ETS는 선물·파생상품 중심, K-ETS는 장내 현물 비중과 유동성 부족이 존재한다. | K-ETS 장내 현물 데이터에서 날짜별 전체 거래 VWAP를 계산한다. |
| `docs/reference/가격_모델.md` | K-ETS 빈티지는 이월·차입 규정 및 이행기간 종료와 연결된다. | 빈티지 겹침과 가격 범위를 보조 컬럼으로 남겨 EDA에서 확인한다. |
| `docs/reference/피처_엔지니어링.md` | 탄소 가격 발견은 현물과 선물의 상호작용에 의해 복잡해질 수 있다. | 가격 계열 선택과 외생 변수 결합 주기를 함께 검토한다. |


## 6. K-ETS VWAP 타겟 후보와 현재 선택

선행연구 사례와 K-ETS 데이터 구조를 함께 고려하면, 현재 단계에서 비교해야 할 타겟 구성 방식은 다음과 같다.

| 후보 | 정의 | 현재 처리 | 장점 | 주요 한계 | 검증 항목 |
|---|---|---|---|---|---|
| `market_vwap` | 같은 날짜의 모든 KAU vintage 거래대금 합계를 거래량 합계로 나눈다. | 주 타겟 가격 계열로 사용한다. | 날짜별 시장 가격을 하나로 정의하고 임의의 winner-takes-one 선택을 피한다. | blended day에서 vintage 구성 변화가 가격에 반영될 수 있다. | `is_blended`, `n_vintages`, `vwap_range`, target 분포 |
| `same_vintage_panel` | 같은 `isu_code` 내부에서만 30일/60일 후 VWAP 로그수익률을 계산한다. | 강건성·진단용 비교 축으로 둔다. | 빈티지 전환 효과를 가격 수익률에서 분리한다. | 빈티지별 표본 수가 줄어든다. | 빈티지별 표본 수, 실제 경과일 분포, 극단 수익률 |
| `volume_confirmed_roll` | 다음 vintage의 거래량 우위가 rolling window와 확인일 조건을 만족하면 대표 vintage를 전환한다. | 후속 민감도 분석 후보로 둔다. | 유동성 이동을 반영한 단일 vintage 시계열을 만들 수 있다. | window와 확인일 수에 민감하다. | rolling 거래량 역전 시점, roll 전후 가격 점프, 되돌림 발생 여부 |
| `calendar_roll` | 연도 또는 이행주기 기준으로 대표 vintage를 고정 전환한다. | 후속 민감도 분석 후보로 둔다. | 설명과 재현이 쉽다. | 실제 유동성 이동과 다를 수 있다. | 고정 전환일 전후 거래량·VWAP 변화 |
| `daily_max_volume` | 매일 거래량이 가장 큰 vintage를 대표로 선택한다. | 진단 지표로만 사용한다. | 계산이 쉽고 전환 문제를 빠르게 드러낸다. | 대표 vintage가 앞뒤로 바뀔 수 있다. | 되돌림 횟수, 빈티지 변경일 가격 점프 |

현재 후속 노트북 흐름은 `market_vwap`를 기준으로 타겟 파일을 만든다. `daily_max_volume`은 타겟 후보에서 제외하고, 빈티지 전환 위험을 설명하는 진단 항목으로만 해석한다.


## 7. 30일/60일 예측 기간과 보조 컬럼

월별 또는 저빈도 외생 변수와의 정합성을 고려해 주력 타겟 예측 기간은 30일과 60일로 둔다. 수익률은 기준일 `market_vwap` 대비 미래 `market_vwap`의 로그수익률로 정의한다.

| 컬럼 | 의미 |
|---|---|
| `market_vwap` | 기준일 전체 거래 VWAP |
| `future_date_30d`, `future_date_60d` | 목표 기간 이후 처음 관측된 미래 거래일 |
| `future_vwap_30d`, `future_vwap_60d` | 미래 거래일의 `market_vwap` |
| `actual_elapsed_days_30d`, `actual_elapsed_days_60d` | 기준일과 미래 거래일 사이의 실제 경과 일수 |
| `target_logret_30d`, `target_logret_60d` | 기준일 대비 미래 `market_vwap` 로그수익률 |
| `same_panel_30d`, `same_panel_60d` | 동일한 날짜별 `market_vwap` 패널 안에서 미래값을 찾았는지 표시하는 컬럼 |
| `n_vintages`, `vintages`, `is_blended` | 기준일에 가격 계산에 들어간 vintage 수, 목록, 혼합 여부 |
| `future_n_vintages_*`, `future_vintages_*`, `future_is_blended_*` | 미래 가격 계산에 들어간 vintage 수, 목록, 혼합 여부 |
| `vwap_range` | 기준일 vintage별 VWAP 최댓값과 최솟값의 차이 |

`02_target_eda.ipynb`에서는 `nominal_plus_15` 기준으로 `pass_30d`, `pass_60d`를 만들어 모델링 표본을 점검한다. 이후 feature 노트북은 이 품질 필터를 기준으로 coverage, as-of join, IC를 확인한다.
